### Running PageIndex Locally

In [52]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.home() / "Projects" / "RAG_v1" / ".env")
NVIDIA_API_KEY = os.environ["NVIDIA_API_KEY"]

import pageindex.utils as _u

# 1) Don't hammer NVIDIA's free tier → no 429s
_u.SUMMARY_CONCURRENCY = 2

# 2) Cap oversized summary prompts (PORT_PACKAGE has a 200k-char node) → no 504s
_orig_acomp = _u.llm_acompletion


async def _capped(model, prompt):
    if len(prompt) > 6000:
        prompt = prompt[:6000] + "\n...[truncated]"
    return await _orig_acomp(model, prompt)


_u.llm_acompletion = _capped

from pageindex import PageIndexLocalClient, utils

pi_client = PageIndexLocalClient(
    model="nvidia_nim/meta/llama-3.1-8b-instruct",
    summary_model="nvidia_nim/meta/llama-3.1-8b-instruct",
    retrieve_model="nvidia_nim/meta/llama-3.1-8b-instruct",
    storage_path=".pageindex",
)
print(pi_client.__class__.__name__)


PageIndexLocalClient


### Building The JSON Tree

In [53]:
PDF_SOURCE_DIR = "Policy Documents Curated 15"
REGISTRY_PATH = "pdf_registry.json"

pdf_files = sorted(
    os.path.join(PDF_SOURCE_DIR, f)
    for f in os.listdir(PDF_SOURCE_DIR)
    if f.lower().endswith(".pdf")
)
print(f"Found {len(pdf_files)} PDFs")

existing = {d["name"]: d["id"] for d in pi_client.list_documents()["documents"]}
doc_ids = {}
for pdf_path in pdf_files:
    filename = os.path.basename(pdf_path)
    if filename in existing:
        doc_ids[filename] = existing[filename]
        print(f"⏭️  Already indexed: {filename}")
        continue
    doc_ids[filename] = pi_client.submit_document(pdf_path)["doc_id"]  # synchronous
    print(f"✅ Submitted: {filename} → {doc_ids[filename]}")

print("\nAll doc_ids:", doc_ids)

Found 15 PDFs
⏭️  Already indexed: 14..Trade credit insc_GEN756.pdf
⏭️  Already indexed: Arogya_Sanjeevani_Policy_Wording_KEN_034037d936.pdf
⏭️  Already indexed: Auto_Secure_Commercial_Vehicle_Package_Policy_Base_Policy_Wording_22b7905015.pdf
⏭️  Already indexed: Bharat_Griha_Raksha_Policy_Policy_Wordings_5219f40e18.pdf
⏭️  Already indexed: Click-2-Protect-Optima-Secure-Policy-Bond-101Y122V05.pdf
⏭️  Already indexed: Cyber_Shield_Policy_Wordings_78baa23b5a.pdf
⏭️  Already indexed: PORT_PACKAGE_Policy_wording_a5f317091b.pdf
⏭️  Already indexed: Policy_Wordings_aviation_insurance.pdf_7b7e60200a.pdf
⏭️  Already indexed: Policy_Wordings_contractors_plant_and_machinery_insurance.pdf_0175bcd067.pdf
⏭️  Already indexed: Policy_Wordings_political_risk_insurance_for_investors.pdf_b7a0e7c805.pdf
⏭️  Already indexed: SBI_General_Livestock_Policy_Wording_38ef0b201b.pdf
⏭️  Already indexed: Weather_Insurance_Policy_Wordings_Retail_6a1bd806a7.pdf
⏭️  Already indexed: hdfc-life-smart-pension-plus-v13

In [54]:
import json

print(json.dumps(doc_ids, indent=4))

{
    "14..Trade credit insc_GEN756.pdf": "pi-58068b94fd464bedaf251e3b442d3183",
    "Arogya_Sanjeevani_Policy_Wording_KEN_034037d936.pdf": "pi-7a77210c58344134988c513ce697a93a",
    "Auto_Secure_Commercial_Vehicle_Package_Policy_Base_Policy_Wording_22b7905015.pdf": "pi-66af3e82285e4e24819dcc87fcda1343",
    "Bharat_Griha_Raksha_Policy_Policy_Wordings_5219f40e18.pdf": "pi-a47f79df4ac040f5984d512c4e1225ae",
    "Click-2-Protect-Optima-Secure-Policy-Bond-101Y122V05.pdf": "pi-12de45f887e644009982f039d294eb7e",
    "Cyber_Shield_Policy_Wordings_78baa23b5a.pdf": "pi-6392aba4cb2f45ce8abf2f75871f3e93",
    "PORT_PACKAGE_Policy_wording_a5f317091b.pdf": "pi-da50dbffde964da4ba36ede6c7079e11",
    "Policy_Wordings_aviation_insurance.pdf_7b7e60200a.pdf": "pi-155f96f798b24b41982378c2bef596a5",
    "Policy_Wordings_contractors_plant_and_machinery_insurance.pdf_0175bcd067.pdf": "pi-9d8ead02fe1f4f6284671bf58c07e80b",
    "Policy_Wordings_political_risk_insurance_for_investors.pdf_b7a0e7c805.pdf": "pi-

In [55]:
d = pi_client.list_documents()["documents"][0]["id"]
t = pi_client.get_tree(d, node_summary=True)["result"]
print(len(t), "top-level nodes")
print(utils.create_node_mapping(t).keys().__len__(), "total nodes")


66 top-level nodes
72 total nodes


### Creating PDF registry for routing

In [56]:
import json

registry = {
    d["id"]: {"doc_id": d["id"], "filename": d["name"], "description": d["description"]}
    for d in pi_client.list_documents()["documents"]
}
with open("pdf_registry.json", "w") as f:
    json.dump(registry, f, indent=2)
print(f"Registry written: {len(registry)} docs, 0 NIM calls")


Registry written: 15 docs, 0 NIM calls


### Calling Nim API

In [66]:
from openai import OpenAI


def call_nim(
    prompt,
    model="meta/llama-3.1-70b-instruct",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
    temperature=0,
    max_tokens=1024,
):
    client = OpenAI(base_url=base_url, api_key=api_key)
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return completion.choices[0].message.content

In [67]:
client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)

models = client.models.list()
for model in models:
    print(model.id)


01-ai/yi-large
adept/fuyu-8b
ai21labs/jamba-1.5-large-instruct
aisingapore/sea-lion-7b-instruct
baai/bge-m3
bigcode/starcoder2-15b
databricks/dbrx-instruct
deepseek-ai/deepseek-coder-6.7b-instruct
deepseek-ai/deepseek-v4-flash-0731
google/codegemma-1.1-7b
google/codegemma-7b
google/deplot
google/diffusiongemma-26b-a4b-it
google/gemma-2b
google/gemma-3-12b-it
google/gemma-3-4b-it
google/gemma-4-31b-it
google/recurrentgemma-2b
ibm/granite-3.0-3b-a800m-instruct
ibm/granite-3.0-8b-instruct
ibm/granite-34b-code-instruct
ibm/granite-8b-code-instruct
meta/codellama-70b
meta/llama-3.1-70b-instruct
meta/llama-3.1-8b-instruct
meta/llama-3.2-11b-vision-instruct
meta/llama-3.2-1b-instruct
meta/llama-3.2-3b-instruct
meta/llama-3.2-90b-vision-instruct
meta/llama-3.3-70b-instruct
meta/llama-guard-4-12b
meta/llama2-70b
meta/muse-glimmer-30b
microsoft/kosmos-2
microsoft/phi-3-vision-128k-instruct
microsoft/phi-3.5-moe-instruct
minimaxai/minimax-m3
mistralai/codestral-22b-instruct-v0.1
mistralai/mistral

In [68]:
response = call_nim(
    prompt="What is the capital of India?",
)
print(response) 

The capital of India is New Delhi.


### Query Router

In [71]:
def route_query(query: str) -> dict:
    with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
        registry = json.load(f)

    catalog = [
        {
            "doc_id": v["doc_id"],
            "filename": v["filename"],
            "description": v["description"],
        }
        for v in registry.values()
    ]

    prompt = prompt = f"""You are a document router for an insurance Q&A assistant.
A user has asked a question. Decide which documents to search and classify the question type.

User question: {query}

Available documents:
{json.dumps(catalog, indent=2)}

Classify the question into ONE of these types:
- "general"    : conceptual, definitional, or comparative questions ("what is X?", "explain Y", "how does Z work", "difference between A and B"). Answerable from general knowledge — EVEN IF one of the documents happens to cover that topic.
- "specific"   : the user asks about a concrete provision, condition, claim, or coverage detail of a policy — the kind of thing you'd only find in the actual wording (grace periods, exclusions, claim process, deductibles, coverage limits).
- "ambiguous"  : too vague to route (e.g. "tell me about my policy").

Reply ONLY with this JSON, nothing else:
{{
    "thinking": "<your reasoning>",
    "type": "specific",
    "doc_ids": ["doc_id_1"],
    "clarification": ""
}}

Rules:
- type "general"   → doc_ids must be [], clarification must be ""
- type "ambiguous" → doc_ids must be [], clarification must be a follow-up question
- type "specific"  → clarification must be ""
- Rule of thumb: if the answer wouldn't change between documents, it's "general". Only search documents when the user needs THEIR policy's actual terms.
- For "specific": list at most the 3 most relevant doc_ids, never more.
"""

    result = call_nim(prompt)
    if result is None:
        raise ValueError("call_nim returned no response")
    parsed = json.loads(result)

    print(f"🔍 Type     : {parsed['type']}")
    print(f"📄 Doc IDs  : {parsed['doc_ids']}")
    print(f"💭 Reasoning: {parsed['thinking']}")

    return parsed


In [72]:
# Test all three types
route_query("What happens if I miss a premium payment?")  # should be specific
route_query("What is term insurance?")  # should be general
route_query("Tell me about my policy")  # should be ambiguous


🔍 Type     : specific
📄 Doc IDs  : ['pi-2e78d54ee6db45d3981bd10064b9de2c', 'pi-ddd505b254324c3ebfe0175b0559b7fe', 'pi-12de45f887e644009982f039d294eb7e']
💭 Reasoning: The user is asking about the consequences of missing a premium payment, which is a specific provision or condition of a policy. This type of information is typically found in the policy wording and may vary between different policies. Therefore, the question is classified as 'specific'. The most relevant documents to search would be those that outline the terms and conditions of various insurance policies.
🔍 Type     : general
📄 Doc IDs  : []
💭 Reasoning: The user is asking for a general definition of term insurance, which is a type of life insurance policy. This type of question is answerable from general knowledge and does not require searching specific policy documents.
🔍 Type     : ambiguous
📄 Doc IDs  : []
💭 Reasoning: The user's question is too vague to determine which policy they are referring to, and the question d

{'thinking': "The user's question is too vague to determine which policy they are referring to, and the question does not contain enough information to narrow down the search to a specific policy.",
 'type': 'ambiguous',
 'doc_ids': [],
 'clarification': 'Could you please provide more information about your policy, such as the type of insurance or the provider?'}

### Search nodes in relevant documents

In [73]:
def search_nodes(doc_id: str, query: str) -> list[str]:
    """
    Searches the node tree of a single PDF for content relevant to the query.
    Returns a list of text chunks tagged with source filename and page number.
    """
    if not pi_client.is_retrieval_ready(doc_id):
        print(f"⚠️  Doc {doc_id} not ready — skipping.")
        return []

    with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
        registry = json.load(f)

    filename = registry[doc_id]["filename"]

    # fetch tree for this specific doc
    tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
    tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])

    prompt = f"""You are given a question and a tree structure of a document.
Each node contains a node id, title, and summary.
Find all nodes likely to contain the answer to the question.

Question: {query}

Document tree:
{json.dumps(tree_without_text, indent=2)}

Reply ONLY with this JSON:
{{
    "thinking": "<your reasoning>",
    "node_list": ["node_id_1", "node_id_2"]
}}
"""

    response_text = call_nim(prompt)
    if response_text is None:
        raise ValueError("call_nim returned no response")
    result = json.loads(response_text)
    
    node_map = utils.create_node_mapping(tree)

    chunks = []
    for node_id in result["node_list"]:
        if node_id not in node_map:
            continue
        node = node_map[node_id]
        chunks.append(
            f"[Source: {filename}, Page {node['page_index']}]\n{node['text']}"
        )

    print(f"  📑 {filename}: {len(chunks)} relevant node(s) found")
    return chunks


In [74]:
def ask(query: str):
    print(f"\n{'=' * 60}")
    print(f"Query: {query}")
    print("=" * 60)

    # Step 1: route — classify query and get relevant doc_ids
    routing = route_query(query)
    q_type = routing["type"]

    # Step 2: branch based on question type
    if q_type == "ambiguous":
        print(f"\nCould you clarify: {routing['clarification']}")
        return

    elif q_type == "general":
        print("\nGeneral question — answering from LLM knowledge.\n")
        answer = call_nim(f"""Answer this insurance question in simple plain language.
Start with a one-sentence summary. Avoid jargon.
Question: {query}""")
        utils.print_wrapped(answer)

    elif q_type == "specific":
        print(f"\nSearching {len(routing['doc_ids'])} document(s)...")

        # Step 3: search nodes in each routed doc and collect chunks
        all_chunks = []
        for doc_id in routing["doc_ids"]:
            chunks = search_nodes(doc_id, query)
            all_chunks.extend(chunks)

        if not all_chunks:
            print("No relevant content found.")
            return

        context = "\n\n---\n\n".join(all_chunks)

        # Step 4: generate answer from retrieved context
        answer = call_nim(f"""Answer the question based only on the context below.
If context comes from multiple documents, mention which document each point is from.

Question: {query}

Context:
{context}

Instructions:
- Use plain simple language, avoid legal jargon
- Use "you" and "your" instead of "the policyholder"
- Start with a one-sentence summary
- End with "Bottom line:" telling the user what to actually do or know
""")
        print("\n📝 Answer:\n")
        utils.print_wrapped(answer) 

In [75]:
ask("What happens if I miss a premium payment?")



Query: What happens if I miss a premium payment?
🔍 Type     : specific
📄 Doc IDs  : ['pi-2e78d54ee6db45d3981bd10064b9de2c', 'pi-ddd505b254324c3ebfe0175b0559b7fe', 'pi-12de45f887e644009982f039d294eb7e']
💭 Reasoning: The user is asking about the consequences of missing a premium payment, which is a specific provision or condition of a policy. This type of information is typically found in the policy wording and may vary between different policies. Therefore, the question is classified as 'specific'. The most relevant documents to search would be those that outline the terms and conditions of various insurance policies.

Searching 3 document(s)...
  📑 tata_aig_travel_insurance_international_plus_health_policy_wordings_012c2ec139.pdf: 69 relevant node(s) found
  📑 specie_insurance_policy_wordings_8d69ba9234.pdf: 3 relevant node(s) found
  📑 Click-2-Protect-Optima-Secure-Policy-Bond-101Y122V05.pdf: 3 relevant node(s) found

📝 Answer:

If you miss a premium payment, your policy will be cons